<div dir="rtl" align="right">
<h2>Fine-tuning محدود همراه با اجرای کنترل</h2>
<p>GPU را فعال کن، <span dir="ltr">Run all</span> را بزن و Google Drive را وصل کن. این نوت‌بوک به سلول‌های قبلی وابسته نیست.</p>
<p>نقطهٔ شروع، همان مدل‌های اصلی ذخیره‌شده با seedهای 42، 43 و 44 است. برای هر seed دو ادامهٔ آموزشی انجام می‌شود: CNN ثابت با learning rate پایین، و باز شدن فقط چهار کانولوشن آخر CNN با همان learning rate. در مجموع شش اجرای جدید و حداکثر ۱۲ epoch برای هر اجرا انجام می‌شود.</p>
<p>هر دو حالت از یک مدل و ضرایب loss یکسان شروع می‌کنند، learning rate برابر <code dir="ltr">1e-5</code> دارند و همان ترتیب تصاویر و flipها را دریافت می‌کنند. BatchNormهای encoder و سایر لایه‌های آن ثابت می‌مانند.</p>
<p>انتخاب checkpoint فقط با validation و همان اولویت قبلی انجام می‌شود. مدل شروع هم نامزد انتخاب است؛ اگر پیشرفتی دیده نشود، همان مدل حفظ می‌شود. این آزمایش، فرضیهٔ بعدی برای بهبود است و نتیجهٔ بهتر را تضمین نمی‌کند.</p>
</div>


In [ ]:
#@title ۱. اتصال به Drive و بررسی GPU
from pathlib import Path
import tensorflow as tf
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title ۲. تعریف توابع مستقل
from pathlib import Path


import hashlib


import json


import math


import os


import platform


import numpy as np


import pandas as pd


from PIL import Image


import tensorflow as tf


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class SpatialPositionEmbedding(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.positions = self.add_weight(
            name="positions", shape=(1, int(input_shape[1]), int(input_shape[2])),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02), trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        return inputs + self.positions


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class ProbabilityToLogits(tf.keras.layers.Layer):
    def __init__(self, binary=False, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.binary, self.epsilon = binary, epsilon

    def call(self, inputs):
        p = tf.clip_by_value(inputs, self.epsilon, 1.0-self.epsilon)
        return tf.math.log(p)-tf.math.log1p(-p) if self.binary else tf.math.log(p)

    def get_config(self):
        return {**super().get_config(), "binary": self.binary, "epsilon": self.epsilon}


@tf.keras.utils.register_keras_serializable(package="BUSI_Training")
class BoundedLogWeight(tf.keras.constraints.Constraint):
    def __call__(self, value):
        return tf.clip_by_value(value, -3.0, 3.0)

    def get_config(self):
        return {}


def soft_dice_parts(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    true_area = tf.reduce_sum(y_true, axis=(1, 2, 3))
    predicted_area = tf.reduce_sum(y_pred, axis=(1, 2, 3))
    intersection = tf.reduce_sum(y_true*y_pred, axis=(1, 2, 3))
    dice = (2*intersection+1e-6)/(true_area+predicted_area+1e-6)
    return dice, tf.cast(true_area > 0, tf.float32)


def segmentation_loss_per_image(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=(1, 2))
    dice, positive = soft_dice_parts(y_true, y_pred)
    return bce + positive*(1-dice)


@tf.keras.utils.register_keras_serializable(package="BUSI_Training")
class ProposedTrainer(tf.keras.Model):
    """Explicitly track task weights as trainable MODEL variables, not loss locals."""
    def __init__(self, network, class_weights, adaptive_task_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.network = network
        self.class_weights = [float(x) for x in class_weights]
        self.adaptive_task_weights = bool(adaptive_task_weights)
        self.log_class = self.add_weight(
            name="log_class", shape=(), initializer="zeros", trainable=self.adaptive_task_weights,
            constraint=BoundedLogWeight())
        self.log_seg = self.add_weight(
            name="log_seg", shape=(), initializer="zeros", trainable=self.adaptive_task_weights,
            constraint=BoundedLogWeight())
        self.total_metric = tf.keras.metrics.Mean(name="loss")
        self.class_metric = tf.keras.metrics.Mean(name="classification_loss")
        self.plain_ce_metric = tf.keras.metrics.Mean(name="classification_plain_ce")
        self.seg_metric = tf.keras.metrics.Mean(name="segmentation_loss")
        self.accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name="classification_accuracy")
        self.dice_metric = tf.keras.metrics.Mean(name="positive_soft_dice")

    @property
    def metrics(self):
        return [self.total_metric, self.class_metric, self.plain_ce_metric,
                self.seg_metric, self.accuracy_metric, self.dice_metric]

    def build(self, input_shape):
        if not self.network.built:
            self.network.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, training=False):
        return self.network(inputs, training=training)

    def _terms(self, targets, predictions):
        labels = tf.cast(tf.reshape(targets["classification"], [-1]), tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions["classification"])
        weighted_ce = ce*tf.gather(tf.constant(self.class_weights, tf.float32), labels)
        seg = segmentation_loss_per_image(targets["segmentation"], predictions["segmentation"])
        # Chosen adaptive weighting objective; NOT a clinical uncertainty estimate.
        if self.adaptive_task_weights:
            total = (tf.exp(-self.log_class)*tf.reduce_mean(weighted_ce)+self.log_class
                     + tf.exp(-self.log_seg)*tf.reduce_mean(seg)+self.log_seg)
        else:
            total = tf.reduce_mean(weighted_ce)+tf.reduce_mean(seg)
        if self.losses:
            total += tf.add_n(self.losses)
        tf.debugging.assert_all_finite(total, "Non-finite training objective")
        return total, weighted_ce, ce, seg

    def _update(self, targets, predictions, terms):
        total, weighted_ce, ce, seg = terms
        self.total_metric.update_state(total, sample_weight=tf.cast(tf.shape(ce)[0], tf.float32))
        self.class_metric.update_state(weighted_ce)
        self.plain_ce_metric.update_state(ce)
        self.seg_metric.update_state(seg)
        self.accuracy_metric.update_state(targets["classification"], predictions["classification"])
        dice, positive = soft_dice_parts(targets["segmentation"], predictions["segmentation"])
        self.dice_metric.update_state(dice, sample_weight=positive)
        return {metric.name: metric.result() for metric in self.metrics}

    def train_step(self, data):
        images, targets, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        if sample_weight is not None:
            raise ValueError("Use the recorded class weights; external sample weights are not supported.")
        with tf.GradientTape() as tape:
            predictions = self(images, training=True)
            terms = self._terms(targets, predictions)
        gradients = tape.gradient(terms[0], self.trainable_variables)
        pairs = [(gradient, variable) for gradient, variable in
                 zip(gradients, self.trainable_variables) if gradient is not None]
        self.optimizer.apply_gradients(pairs)
        return self._update(targets, predictions, terms)

    def test_step(self, data):
        images, targets, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        predictions = self(images, training=False)
        return self._update(targets, predictions, self._terms(targets, predictions))

    def get_config(self):
        return {**super().get_config(),
                "network": tf.keras.utils.serialize_keras_object(self.network),
                "class_weights": self.class_weights,
                "adaptive_task_weights": self.adaptive_task_weights}

    @classmethod
    def from_config(cls, config):
        config = dict(config)
        network = tf.keras.utils.deserialize_keras_object(config.pop("network"))
        return cls(network=network, **config)


def validation_statistics(labels, masks, probabilities):
    """Image-level validation metrics. All class orders: normal, benign, malignant."""
    labels = np.asarray(labels, dtype=np.int32)
    predicted = np.argmax(probabilities["classification"], axis=1)
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (labels, predicted), 1)
    tp = np.diag(matrix).astype(float)
    precision = np.divide(tp, matrix.sum(0), out=np.zeros(3), where=matrix.sum(0)>0)
    recall = np.divide(tp, matrix.sum(1), out=np.zeros(3), where=matrix.sum(1)>0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros(3), where=precision+recall>0)
    malignant = labels == 2
    normal = labels == 0
    false_malignant = int(((predicted == 2) & ~malignant).sum())
    specificity = 1-false_malignant/int((~malignant).sum())
    true_masks = np.asarray(masks)[..., 0] > 0
    predicted_masks = np.asarray(probabilities["segmentation"])[..., 0] >= 0.5
    true_area = true_masks.sum((1, 2))
    predicted_area = predicted_masks.sum((1, 2))
    intersection = (true_masks & predicted_masks).sum((1, 2))
    positive = true_area > 0
    dice = np.full(len(labels), np.nan)
    dice[positive] = 2*intersection[positive]/(true_area[positive]+predicted_area[positive])
    stats = {
        "val_accuracy": float(tp.sum()/len(labels)),
        "val_macro_f1": float(f1.mean()),
        "val_malignant_recall": float(recall[2]),
        "val_missed_malignant": int((predicted[malignant] != 2).sum()),
        "val_nonmalignant_predicted_malignant": false_malignant,
        "val_nonmalignant_specificity": float(specificity),
        "val_malignant_balanced_accuracy": float((recall[2]+specificity)/2),
        "val_normal_recall": float(recall[0]),
        "val_positive_dice": float(dice[positive].mean()),
        "val_malignant_dice": float(dice[malignant].mean()),
        "val_very_low_dice": int((dice[positive] < 0.1).sum()),
        "val_empty_positive_predictions": int((predicted_area[positive] == 0).sum()),
        "val_normal_with_any_foreground": int((predicted_area[normal] > 0).sum()),
        "val_normal_mean_foreground_percent": float(
            100*predicted_area[normal].mean()/true_masks.shape[1]/true_masks.shape[2]),
        "val_normal_max_foreground_percent": float(
            100*predicted_area[normal].max()/true_masks.shape[1]/true_masks.shape[2])}
    return stats, matrix, dice, predicted, predicted_area


def _write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf8")
    os.replace(tmp, path)


def _save_model(model, path):
    path = Path(path)
    tmp = path.with_name(path.stem+".tmp.keras")
    model.save(str(tmp))
    os.replace(tmp, path)


def _selection_tuple(stats):
    # Predeclared: primary malignant-vs-other balanced accuracy; ties: Dice, macro-F1.
    return (stats["val_malignant_balanced_accuracy"],
            stats["val_positive_dice"], stats["val_macro_f1"])


class ValidationCheckpoints(tf.keras.callbacks.Callback):
    def __init__(self, directory, data, patience, initial_epoch):
        super().__init__()
        self.directory, self.data = Path(directory), data
        self.patience, self.wait = patience, 0
        self.initial_epoch = initial_epoch
        self.best = None
        self.progress = self.directory/"progress.json"
        self.best_metadata = self.directory/"best_validation.json"

    def _predict(self):
        probabilities = self.model.network.predict(self.data["X_val"], batch_size=8, verbose=0)
        for values in probabilities.values():
            if not np.isfinite(values).all():
                raise FloatingPointError("Non-finite validation predictions")
        return validation_statistics(self.data["y_val"], self.data["M_val"], probabilities)[0]

    def _save_best(self, stats, epoch_number):
        _save_model(self.model.network, self.directory/"best.keras")
        _save_model(self.model, self.directory/"best_trainer.keras")
        _write_json(self.best_metadata, {"epoch_number": epoch_number, **stats})
        self.best = _selection_tuple(stats)

    def on_train_begin(self, logs=None):
        if self.best_metadata.exists():
            self.best = _selection_tuple(json.loads(self.best_metadata.read_text()))
        else:
            initial = self._predict()
            self._save_best(initial, self.initial_epoch)
            _write_json(self.directory/"initial_validation.json", initial)
        if self.progress.exists():
            progress = json.loads(self.progress.read_text())
            if progress.get("completed_epochs") == self.initial_epoch:
                self.wait = int(progress.get("wait", 0))

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            raise RuntimeError("Epoch logs missing")
        stats = self._predict()
        logs.update(stats)
        logs["task_weight_classification"] = float(tf.exp(-self.model.log_class).numpy())
        logs["task_weight_segmentation"] = float(tf.exp(-self.model.log_seg).numpy())
        if not np.isfinite(float(logs["loss"])):
            raise FloatingPointError("Non-finite objective")
        score = _selection_tuple(stats)
        if score > self.best:
            self._save_best(stats, epoch+1)
            self.wait = 0
        else:
            self.wait += 1
        record = {"epoch": epoch, **{k: float(v) for k, v in logs.items()}}
        _write_json(self.directory/"epochs"/f"{epoch:04d}.json", record)
        _save_model(self.model, self.directory/"last_trainer.keras")
        _write_json(self.progress, {"completed_epochs": epoch+1, "wait": self.wait})
        print("\nValidation: malignant recall {:.3f}, false malignant {}, Dice {:.3f}, task weights {:.3f}/{:.3f}".format(
            stats["val_malignant_recall"], stats["val_nonmalignant_predicted_malignant"],
            stats["val_positive_dice"], logs["task_weight_classification"], logs["task_weight_segmentation"]))
        if self.wait >= self.patience:
            self.model.stop_training = True
            print("Early stopping: no improvement under the declared selection rule.")


def _read_data(proposed_dir, manifest, root, data):
    frames = {split: manifest[manifest["split"] == split].reset_index(drop=True)
              for split in ["train", "val"]}
    labels = {"normal": 0, "benign": 1, "malignant": 2}
    source = json.loads((proposed_dir/"source_config.json").read_text())
    stats = source["normalization"]
    mean, std = np.array(stats["mean"], np.float32), np.array(stats["std"], np.float32)
    if data is None:
        print("Restoring only train/validation images and masks; no previous training is rerun.")
        data = {}
        for split, frame in frames.items():
            images = np.empty((len(frame), 224, 224, 3), np.float32)
            masks = np.empty((len(frame), 224, 224, 1), np.float32)
            for i, row in frame.iterrows():
                with Image.open(root/row["image"]) as image:
                    images[i] = np.asarray(image.convert("RGB").resize((224,224), Image.Resampling.BILINEAR))
                with Image.open(root/row["mask"]) as image:
                    masks[i,...,0] = np.asarray(image.convert("L").resize((224,224), Image.Resampling.NEAREST)) > 0
                if i % 100 == 0:
                    print(split, i+1, "/", len(frame))
            data["X_"+split] = (images-mean)/std
            data["M_"+split] = masks
            data["y_"+split] = frame["class_name"].map(labels).to_numpy(dtype=np.int32)
    else:
        data = dict(data)
        print("Using the train/validation arrays already in memory.")
    for split, frame in frames.items():
        n = len(frame)
        assert data["X_"+split].shape == (n,224,224,3)
        assert data["M_"+split].shape == (n,224,224,1)
        assert np.array_equal(data["y_"+split], frame["class_name"].map(labels).to_numpy())
        assert np.isfinite(data["X_"+split]).all()
        assert np.isin(data["M_"+split], [0,1]).all()
        assert np.array_equal(data["M_"+split].sum((1,2,3)) > 0, data["y_"+split] != 0)
    assert np.allclose(data["X_train"].mean((0,1,2), dtype=np.float64), 0, atol=0.01), "Expected training Z-score inputs."
    return data, frames


import gc


import shutil


import tempfile


import matplotlib.pyplot as plt


REPAIR_NAMES = ["normal", "benign", "malignant"]


REPAIR_FIELDS = ["val_accuracy", "val_macro_f1", "val_malignant_recall",
    "val_nonmalignant_predicted_malignant", "val_malignant_balanced_accuracy",
    "val_positive_dice", "val_malignant_dice"]


def _repair_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _repair_guard(path, value):
    if path.is_file():
        if json.loads(path.read_text()) != value:
            raise ValueError("این پوشه مربوط به تنظیمات دیگری است؛ خروجی قبلی را حفظ کن: "+str(path))
    else:
        _write_json(path, value)


def _repair_prediction(network, data):
    values = network.predict(data["X_val"],batch_size=8,verbose=0)
    if not isinstance(values,dict) or set(values) != {"classification","segmentation"}:
        raise ValueError("Expected named classification/segmentation outputs")
    p,mask = [np.asarray(values[key]) for key in ["classification","segmentation"]]
    if p.shape != (115,3) or mask.shape != (115,224,224,1):
        raise ValueError("Unexpected validation output shapes")
    if not np.isfinite(p).all() or not np.isfinite(mask).all() or (p<0).any() or (p>1).any() or (mask<0).any() or (mask>1).any():
        raise FloatingPointError("Invalid validation probabilities")
    np.testing.assert_allclose(p.sum(1),1,atol=1e-5,rtol=0)
    return values


def _repair_per_image(frame, data, values, size_cuts):
    stats,matrix,dice,predicted,predicted_area = validation_statistics(data["y_val"],data["M_val"],values)
    truth = data["y_val"]
    p = np.asarray(values["classification"])
    area = data["M_val"].sum((1,2,3))
    result = frame[["image","mask","class_name","group_id"]].copy()
    result["predicted_class"] = [REPAIR_NAMES[i] for i in predicted]
    result["classification_error"] = predicted != truth
    result["confidence"] = p.max(1)
    result["probability_of_true_class"] = p[np.arange(len(truth)),truth]
    for i,name in enumerate(REPAIR_NAMES):
        result["p_"+name] = p[:,i]
    result["reference_pixels"] = area.astype(int)
    result["reference_area_percent"] = area*100/(224*224)
    result["predicted_pixels"] = predicted_area
    result["dice"] = dice
    result["normal_false_foreground_percent"] = np.where(truth==0,predicted_area*100/(224*224),np.nan)
    result["normal_class_with_mask"] = (predicted==0) & (predicted_area>0)
    result["lesion_class_without_mask"] = (predicted!=0) & (predicted_area==0)
    stats["val_normal_class_with_mask"] = int(result["normal_class_with_mask"].sum())
    stats["val_lesion_class_without_mask"] = int(result["lesion_class_without_mask"].sum())
    stats["val_inconsistent_pairs"] = stats["val_normal_class_with_mask"]+stats["val_lesion_class_without_mask"]
    bins = np.searchsorted(size_cuts,area/(224*224),side="right")
    result["lesion_size_bin"] = ["no_lesion" if a==0 else "Q"+str(int(b)+1) for a,b in zip(area,bins)]
    return stats,matrix,result


def _repair_save_predictions(folder, frame, data, values, size_cuts, condition, seed, epoch):
    folder.mkdir(parents=True,exist_ok=True)
    stats,matrix,per_image = _repair_per_image(frame,data,values,size_cuts)
    per_image.to_csv(folder/"validation_per_image.csv",index=False)
    pd.DataFrame(matrix,index=REPAIR_NAMES,columns=REPAIR_NAMES).to_csv(folder/"validation_confusion.csv")
    binary = np.asarray(values["segmentation"])[...,0] >= 0.5
    temporary = folder/"validation_predictions.tmp.npz"
    np.savez_compressed(temporary,probabilities=np.asarray(values["classification"],np.float32),
        mask_bits=np.packbits(binary.reshape(len(binary),-1),axis=1),
        images=frame["image"].to_numpy(dtype=str),mask_shape=np.array([224,224],np.int32))
    os.replace(temporary,folder/"validation_predictions.npz")
    row = {"condition":condition,"seed":seed,"selected_epoch":int(epoch),**stats}
    _write_json(folder/"summary.json",row)
    return row,per_image


def _repair_check_baseline_config(config, initial_sha, weights):
    expected = {"source_sha256":initial_sha,"class_order":REPAIR_NAMES,
        "class_cost_ratio":[1,1,2],"cnn_encoder_trainable":False,
        "optimizer":"Adam","learning_rate":1e-4,"clipnorm":1.0,
        "batch_size":8,"max_epochs":30,"patience":8,
        "augmentation":"paired_horizontal_flip_probability_0.5",
        "selection_priority":["malignant_vs_others_balanced_accuracy","positive_hard_dice","macro_f1"],
        "adaptive_loss":"exp(-s_cls)*weighted_CE+s_cls+exp(-s_seg)*seg_loss+s_seg",
        "adaptive_log_weight_bounds":[-3,3],"initial_task_weights":[1,1],
        "segmentation_loss":"BCE_all_plus_Dice_on_positive_images"}
    different = {k:(config.get(k),v) for k,v in expected.items() if config.get(k) != v}
    if different:
        raise ValueError("تنظیمات کنترل با این آزمایش همسان نیست: "+str(different))
    np.testing.assert_allclose(config["normalized_class_weights"],weights,atol=1e-6,rtol=0)


def _repair_choose_protocol(experiments_root, evaluation_dir):
    if evaluation_dir is not None:
        return Path(evaluation_dir)
    found = sorted(Path(experiments_root).glob("proposed_model_*/ablations/component_study_v1/final_evaluation_v1/frozen_protocol.json"))
    if not found:
        raise FileNotFoundError("پروژهٔ ارزیابی نهایی قبلی پیدا نشد: "+str(experiments_root))
    for i,path in enumerate(found):
        print(f"[{i}] {path.parent}")
    answer = "0" if len(found)==1 else input("شمارهٔ پروژهٔ قبلی را وارد کن: ").strip()
    if not answer.isdigit() or not 0<=int(answer)<len(found):
        raise ValueError("شمارهٔ پروژه معتبر نیست")
    return found[int(answer)].parent


"""Paired low-LR continuation and limited CNN fine-tuning; validation only."""
PHASE_VERSION="1.0.0"
PHASE_MODES=["frozen_low_lr","last4_convs_low_lr"]
PHASE_SEEDS=[42,43,44]


def _phase_leaves(model,prefix=""):
    found=[]
    for layer in model.layers:
        path=prefix+layer.name
        if isinstance(layer,tf.keras.Model):
            found.extend(_phase_leaves(layer,path+"/"))
        else:
            found.append((path,layer))
    return found


def _phase_configure(network,mode):
    if mode not in PHASE_MODES:
        raise ValueError("Unknown fine-tuning mode")
    encoder=network.get_layer("shared_encoder")
    conv_types=(tf.keras.layers.Conv2D,tf.keras.layers.DepthwiseConv2D,tf.keras.layers.SeparableConv2D)
    leaves=_phase_leaves(encoder)
    convolutions=[(name,layer) for name,layer in leaves if isinstance(layer,conv_types)]
    if len(convolutions)<4:
        raise ValueError("Expected at least four encoder convolutions; send the encoder layer listing")
    allowed=[name for name,_ in convolutions[-4:]] if mode=="last4_convs_low_lr" else []
    if mode=="frozen_low_lr":
        encoder.trainable=False
    else:
        encoder.trainable=True
        for name,layer in leaves:
            layer.trainable=name in allowed
    for name,layer in leaves:
        if isinstance(layer,tf.keras.layers.BatchNormalization) and layer.trainable:
            raise RuntimeError("An encoder BatchNormalization layer is trainable")
    actual=[name for name,layer in leaves if layer.trainable and layer.weights]
    if set(actual)!=set(allowed):
        raise RuntimeError("Unexpected trainable encoder layers: "+str(actual))
    return allowed


def _phase_frozen_digest(network,allowed):
    digest=hashlib.sha256()
    for path,layer in _phase_leaves(network.get_layer("shared_encoder")):
        if path in allowed:
            continue
        for weight in layer.weights:
            array=np.ascontiguousarray(weight.numpy())
            digest.update(str((path,weight.name,array.shape,array.dtype.str)).encode())
            digest.update(memoryview(array).cast("B"))
    return digest.hexdigest()


def _phase_trainable_encoder(network,allowed):
    return [w for path,layer in _phase_leaves(network.get_layer("shared_encoder")) if path in allowed for w in layer.trainable_weights]


def _phase_epoch_arrays(data,seed,epoch):
    """Identical shuffling and flips for both conditions in the same seed/epoch."""
    n=len(data["y_train"])
    order=np.random.default_rng(np.random.SeedSequence([seed,epoch,4801])).permutation(n)
    flags=np.random.default_rng(np.random.SeedSequence([seed,epoch,4802])).random(n)<.5
    x,m=data["X_train"][order].copy(),data["M_train"][order].copy()
    x[flags]=x[flags,:,::-1,:]
    m[flags]=m[flags,:,::-1,:]
    return x,m,data["y_train"][order]


def _phase_new_trainer(source_trainer,mode,data,weights):
    saved=tf.keras.models.load_model(str(source_trainer),compile=False)
    np.testing.assert_allclose(saved.class_weights,weights,atol=1e-6,rtol=0)
    if not saved.adaptive_task_weights:
        raise ValueError("Expected the original adaptive task-weight trainer")
    network=saved.network
    if not network.trainable:
        raise ValueError("The source network is unexpectedly frozen at its root")
    allowed=_phase_configure(network,mode)
    trainer=ProposedTrainer(network,weights.tolist(),name="targeted_finetune_trainer")
    trainer(data["X_train"][:1],training=False)
    trainer.log_class.assign(saved.log_class)
    trainer.log_seg.assign(saved.log_seg)
    for value in [trainer.log_class,trainer.log_seg]:
        if not np.isfinite(float(value.numpy())) or abs(float(value.numpy()))>3.00001:
            raise ValueError("Invalid restored task-weight coefficient")
    # A fresh optimizer is necessary when the set of trainable variables changes.
    # Both arms use the same fresh Adam optimizer, LR and starting task weights.
    trainer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5,clipnorm=1.0))
    if not network.trainable_weights:
        raise ValueError("No trainable network weights are available")
    if any(id(w) not in {id(v) for v in network.trainable_weights} for w in _phase_trainable_encoder(network,allowed)):
        raise ValueError("Selected encoder variables are hidden by a frozen parent")
    del saved
    return trainer,allowed


def _phase_smoke(item,data,weights,report):
    records=[]
    for mode in PHASE_MODES:
        tf.keras.utils.set_random_seed(item["seed"])
        trainer,allowed=_phase_new_trainer(item["trainer"],mode,data,weights)
        frozen_before=_phase_frozen_digest(trainer.network,allowed)
        indices=[int(np.flatnonzero(data["y_train"]==i)[0]) for i in range(3)]
        x=data["X_train"][indices]
        target={"classification":data["y_train"][indices],"segmentation":data["M_train"][indices]}
        enc_variables=_phase_trainable_encoder(trainer.network,allowed)
        before=[w.numpy().copy() for w in enc_variables]
        logs=trainer.train_on_batch(x,target,return_dict=True)
        if not all(np.isfinite(float(value)) for value in logs.values()):
            raise FloatingPointError("Non-finite fine-tuning smoke-test loss")
        if _phase_frozen_digest(trainer.network,allowed)!=frozen_before:
            raise RuntimeError("Frozen encoder weights or BatchNorm statistics changed")
        changed=any(not np.array_equal(a,w.numpy()) for a,w in zip(before,enc_variables))
        if mode=="last4_convs_low_lr" and not changed:
            raise RuntimeError("Unfrozen convolution weights did not receive an update")
        predicted=trainer(x,training=False)
        with tempfile.TemporaryDirectory(prefix="busi_phase_smoke_") as directory:
            path=Path(directory)/"trainer.keras"
            _save_model(trainer,path)
            restored=tf.keras.models.load_model(str(path))
            observed=restored(x,training=False)
            for key in predicted:
                np.testing.assert_allclose(predicted[key].numpy(),observed[key].numpy(),atol=1e-6,rtol=1e-6)
            if int(restored.optimizer.iterations.numpy())!=1:
                raise RuntimeError("Optimizer state did not survive reload")
            if _phase_configure(restored.network,mode)!=allowed:
                raise RuntimeError("Trainability configuration changed on reload")
            del restored,observed
        records.append({"mode":mode,"encoder_trainable_layers":allowed,
            "trainable_parameters":sum(int(np.prod(w.shape)) for w in trainer.trainable_weights),
            "frozen_encoder_unchanged":True,"encoder_update_observed":changed,
            "save_reload":"passed","logs":{k:float(v) for k,v in logs.items()}})
        del trainer,predicted,before,enc_variables
        gc.collect()
    _write_json(report/"colab_smoke_test.json",records)
    print("Both continuation/fine-tuning smoke tests passed.",flush=True)


def _phase_run_one(out,report,item,mode,data,frame,size_cuts,weights,reference_values):
    directory=out/mode/f"seed{item['seed']}"
    directory.mkdir(parents=True,exist_ok=True)
    (directory/"epochs").mkdir(exist_ok=True)
    destination=report/f"{mode}_seed{item['seed']}"
    _repair_guard(directory/"settings.json",{"phase_settings_sha256":_repair_sha(out/"settings.json"),
        "source_trainer_sha256":item["trainer_sha256"],"mode":mode,"seed":item["seed"]})
    if (directory/"completed.json").is_file():
        saved=json.loads((directory/"completed.json").read_text())
        if _repair_sha(directory/"best.keras")!=saved["best_sha256"]:
            raise ValueError("A completed fine-tuned checkpoint changed")
        return json.loads((destination/"summary.json").read_text()),pd.read_csv(destination/"validation_per_image.csv")
    tf.keras.utils.set_random_seed(item["seed"])
    fresh,allowed=_phase_new_trainer(item["trainer"],mode,data,weights)
    if fresh.network.count_params()!=item["parameters"]:
        raise ValueError("Fine-tuning unexpectedly changed network parameter count")
    initial_values=_repair_prediction(fresh.network,data)
    for key in initial_values:
        np.testing.assert_allclose(initial_values[key],reference_values[key],atol=1e-5,rtol=0,
            err_msg="Initial predictions differ after restoring the source network")
    frozen_digest=_phase_frozen_digest(fresh.network,allowed)
    _repair_guard(directory/"trainability.json",{"encoder_trainable_layers":allowed,
        "frozen_encoder_sha256":frozen_digest,
        "trainable_parameters":sum(int(np.prod(w.shape)) for w in fresh.trainable_weights),
        "initial_log_class":float(fresh.log_class.numpy()),"initial_log_seg":float(fresh.log_seg.numpy())})
    steps=math.ceil(len(data["y_train"])/8)
    last=directory/"last_trainer.keras"
    if last.is_file():
        del fresh
        gc.collect()
        trainer=tf.keras.models.load_model(str(last))
        actual=[name for name,layer in _phase_leaves(trainer.network.get_layer("shared_encoder")) if layer.trainable and layer.weights]
        if set(actual)!=set(allowed) or _phase_frozen_digest(trainer.network,allowed)!=frozen_digest:
            raise ValueError("Resumed trainability or frozen encoder state differs")
        iterations=int(trainer.optimizer.iterations.numpy())
        if iterations%steps:
            raise ValueError("Checkpoint is not at an epoch boundary")
        start=iterations//steps
        for epoch in range(start):
            if not (directory/"epochs"/f"{epoch:04d}.json").is_file():
                raise FileNotFoundError("Missing completed epoch record")
        print("Resuming",mode,"seed",item["seed"],"after epoch",start,flush=True)
    else:
        trainer=fresh
        start=0
    val=tf.data.Dataset.from_tensor_slices((data["X_val"],{
        "classification":data["y_val"],"segmentation":data["M_val"]})).batch(8)
    monitor=ValidationCheckpoints(directory,data,patience=4,initial_epoch=start)
    for epoch in range(start,12):
        progress=json.loads((directory/"progress.json").read_text()) if (directory/"progress.json").is_file() else {}
        if progress.get("completed_epochs")==epoch and progress.get("wait",0)>=4:
            break
        x,m,y=_phase_epoch_arrays(data,item["seed"],epoch)
        # Fingerprint verifies that paired arms actually receive identical data.
        digest=hashlib.sha256()
        for array in [x,m,y]:
            digest.update(memoryview(np.ascontiguousarray(array)).cast("B"))
        shared=out/"paired_epoch_inputs"/f"seed{item['seed']}_epoch{epoch:02d}.json"
        shared.parent.mkdir(exist_ok=True)
        _repair_guard(shared,{"sha256":digest.hexdigest(),"seed":item["seed"],"phase_epoch":epoch+1})
        train=tf.data.Dataset.from_tensor_slices((x,{"classification":y,"segmentation":m})).batch(8).prefetch(1)
        monitor.initial_epoch=epoch
        trainer.fit(train,validation_data=val,initial_epoch=epoch,epochs=epoch+1,callbacks=[monitor],verbose=2)
        if int(trainer.optimizer.iterations.numpy())!=(epoch+1)*steps:
            raise RuntimeError("Unexpected training step count")
        if _phase_frozen_digest(trainer.network,allowed)!=frozen_digest:
            raise RuntimeError("Frozen encoder weights/BatchNorm changed during training")
        records=[json.loads((directory/"epochs"/f"{e:04d}.json").read_text()) for e in range(epoch+1)]
        pd.DataFrame(records).to_csv(directory/"training_log.csv",index=False)
        del train,x,m,y
        gc.collect()
        if trainer.stop_training:
            break
    del trainer,val,monitor,initial_values
    gc.collect()
    best=tf.keras.models.load_model(str(directory/"best.keras"),compile=False)
    if _phase_frozen_digest(best,allowed)!=frozen_digest:
        raise RuntimeError("Selected model changed the frozen encoder state")
    values=_repair_prediction(best,data)
    metadata=json.loads((directory/"best_validation.json").read_text())
    actual=validation_statistics(data["y_val"],data["M_val"],values)[0]
    for field in REPAIR_FIELDS:
        np.testing.assert_allclose(actual[field],metadata[field],atol=1e-5,rtol=0)
    row,per_image=_repair_save_predictions(destination,frame,data,values,size_cuts,mode,item["seed"],metadata["epoch_number"])
    row["source_epoch"]=item["selected_epoch"]
    row["selected_phase_epoch"]=int(metadata["epoch_number"])
    row["initial_model_retained"]=int(metadata["epoch_number"])==0
    _write_json(destination/"summary.json",row)
    for filename in ["training_log.csv","trainability.json","best_validation.json","initial_validation.json"]:
        if (directory/filename).is_file():
            shutil.copyfile(directory/filename,destination/filename)
    _write_json(directory/"completed.json",{"best_sha256":_repair_sha(directory/"best.keras"),"test_evaluated":False})
    del best,values
    gc.collect()
    return row,per_image


def _phase_final_tables(report,rows,per_images):
    summary=pd.DataFrame(rows)
    summary.to_csv(report/"validation_comparison.csv",index=False)
    fields=["val_accuracy","val_macro_f1","val_malignant_recall","val_nonmalignant_specificity",
        "val_nonmalignant_predicted_malignant","val_malignant_balanced_accuracy","val_positive_dice",
        "val_malignant_dice","val_normal_mean_foreground_percent","val_inconsistent_pairs"]
    means=summary.groupby("condition",sort=False)[fields].mean()
    means.to_csv(report/"validation_means.csv")
    summary.groupby("condition",sort=False)[fields].agg(["mean","std"]).to_csv(report/"validation_mean_std.csv")
    original=summary[summary["condition"]=="original_model"].set_index("seed")
    for mode in PHASE_MODES:
        subset=summary[summary["condition"]==mode].set_index("seed")
        (subset[fields]-original[fields]).to_csv(report/(mode+"_minus_original.csv"))
    tuned=summary[summary["condition"]==PHASE_MODES[1]].set_index("seed")
    frozen=summary[summary["condition"]==PHASE_MODES[0]].set_index("seed")
    (tuned[fields]-frozen[fields]).to_csv(report/"unfreezing_minus_frozen_control.csv")
    comparisons=[]
    for mode in PHASE_MODES:
        old,new=means.loc["original_model"],means.loc[mode]
        criteria={"primary_balanced_accuracy_improved":bool(new["val_malignant_balanced_accuracy"]>old["val_malignant_balanced_accuracy"]+1e-12),
            "malignant_recall_not_lower":bool(new["val_malignant_recall"]>=old["val_malignant_recall"]-1e-12),
            "positive_dice_not_lower":bool(new["val_positive_dice"]>=old["val_positive_dice"]-1e-12),
            "normal_false_area_not_higher":bool(new["val_normal_mean_foreground_percent"]<=old["val_normal_mean_foreground_percent"]+1e-12)}
        comparisons.append({"condition":mode,**criteria,"meets_all_validation_guards":all(criteria.values())})
    guards=pd.DataFrame(comparisons)
    guards.to_csv(report/"validation_improvement_guards.csv",index=False)
    all_images=pd.concat(per_images,ignore_index=True)
    all_images.to_csv(report/"validation_all_images.csv",index=False)
    all_images.loc[all_images["classification_error"]].sort_values("confidence",ascending=False).to_csv(report/"classification_errors.csv",index=False)
    return summary,means,guards


def _phase_figures(report,frame,data,normalization):
    folders=[report/f"{condition}_seed42" for condition in ["original_model"]+PHASE_MODES]
    before=pd.read_csv(folders[0]/"validation_per_image.csv")
    errors=before[before["classification_error"]].sort_values("confidence",ascending=False).head(6).index.tolist()
    normals=before[before["class_name"]=="normal"].sort_values("predicted_pixels",ascending=False).head(3).index.tolist()
    indices=list(dict.fromkeys(errors+normals))
    before.iloc[indices][["image","class_name","group_id"]].to_csv(report/"diagnostic_cases.csv",index=False)
    arrays=[]
    for folder in folders:
        with np.load(folder/"validation_predictions.npz",allow_pickle=False) as a:
            np.testing.assert_array_equal(a["images"],frame["image"].to_numpy())
            masks=np.unpackbits(a["mask_bits"],axis=1,count=224*224).reshape(-1,224,224).astype(bool)
            arrays.append((a["probabilities"].copy(),masks))
    mean,std=[np.asarray(normalization[k],np.float32) for k in ["mean","std"]]
    folder=report/"figures";folder.mkdir(exist_ok=True)
    for number,index in enumerate(indices,1):
        raw=np.clip((data["X_val"][index]*std+mean)/255,0,1)
        fig,axes=plt.subplots(1,4,figsize=(13,3.7))
        axes[0].imshow(raw)
        target=data["M_val"][index,...,0]>0
        if target.any():
            axes[0].imshow(np.ma.masked_where(~target,target),cmap="Reds",vmin=0,vmax=1,alpha=.45)
        axes[0].set_title("Reference: "+frame.iloc[index]["class_name"])
        for col,(label,(p,m)) in enumerate(zip(["Original","Frozen / low LR","Last 4 convs / low LR"],arrays),1):
            axes[col].imshow(raw)
            if m[index].any():
                axes[col].imshow(np.ma.masked_where(~m[index],m[index]),cmap="Blues",vmin=0,vmax=1,alpha=.45)
            axes[col].set_title(label+"\n"+REPAIR_NAMES[int(p[index].argmax())]+f" (p={p[index].max():.3f})",fontsize=9)
        for ax in axes:
            ax.axis("off")
        fig.suptitle(frame.iloc[index]["image"],fontsize=10)
        fig.tight_layout(rect=(0,0,1,.92))
        fig.savefig(folder/f"case_{number:02d}.png",dpi=150,bbox_inches="tight",facecolor="white")
        plt.close(fig)


def run_busi_targeted_finetune(evaluation_dir=None,
        dataset_root="/content/drive/MyDrive/BUSI_MERGED",experiments_root="/content/drive/MyDrive/BUSI_EXPERIMENTS"):
    if not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("GPU را در Colab فعال کن.")
    evaluation_dir=_repair_choose_protocol(experiments_root,evaluation_dir)
    protocol_path=evaluation_dir/"frozen_protocol.json"
    protocol=json.loads(protocol_path.read_text())
    if _repair_sha(protocol_path)!=json.loads((evaluation_dir/"completed.json").read_text())["protocol_sha256"]:
        raise ValueError("Frozen evaluation protocol changed")
    if protocol["class_order"]!=REPAIR_NAMES:
        raise ValueError("Class order changed")
    items=[]
    for seed in PHASE_SEEDS:
        matches=[c for c in protocol["checkpoints"] if c["variant"]=="full_lprm_v1" and c["seed"]==seed]
        if len(matches)!=1:
            raise ValueError("Missing unique original checkpoint")
        item=dict(matches[0]);item["trainer"]=str(Path(item["model"]).parent/"best_trainer.keras")
        if _repair_sha(item["model"])!=item["model_sha256"]:
            raise ValueError("An original model checkpoint changed")
        item["trainer_sha256"]=_repair_sha(item["trainer"])
        items.append(item)
    sources={str(Path(i["model"]).parent.parent) for i in items}
    if len(sources)!=1:
        raise ValueError("Expected one source project")
    source=Path(sources.pop())
    manifest_path=source/"split_manifest.csv"
    if _repair_sha(manifest_path)!=protocol["manifest_sha256"]:
        raise ValueError("Fixed split changed")
    manifest=pd.read_csv(manifest_path)
    if len(manifest)!=761 or not manifest["image"].is_unique or manifest["split"].value_counts().to_dict()!={"train":499,"val":115,"test":147}:
        raise ValueError("Unexpected image split")
    if manifest["group_id"].isna().any() or manifest.groupby("group_id")["split"].nunique().max()!=1:
        raise ValueError("Similarity groups cross splits")
    for column in ["image","mask"]:
        if any(Path(p).is_absolute() or ".." in Path(p).parts for p in manifest[column]):
            raise ValueError("Expected relative dataset paths")
    normalization=json.loads((source/"source_config.json").read_text())["normalization"]
    for key in ["mean","std"]:
        np.testing.assert_allclose(normalization[key],protocol["normalization"][key],atol=1e-7,rtol=0)
    labels=manifest.loc[manifest["split"]=="train","class_name"].map(dict(zip(REPAIR_NAMES,range(3)))).to_numpy(np.int32)
    weights=np.array([1,1,2],np.float32);weights/=weights[labels].mean()
    initial_sha=_repair_sha(source/"initial.keras")
    for item in items:
        cfg=json.loads((Path(item["model"]).parent/"config.json").read_text())
        _repair_check_baseline_config(cfg,initial_sha,weights)
    out=evaluation_dir.parent/"targeted_finetune_v1"
    out.mkdir(exist_ok=True)
    report=out/"reports";report.mkdir(exist_ok=True)
    settings={"version":PHASE_VERSION,"seeds":PHASE_SEEDS,"modes":PHASE_MODES,
        "source_checkpoints":[{k:i[k] for k in ["seed","selected_epoch","model","model_sha256","trainer","trainer_sha256"]} for i in items],
        "protocol_sha256":_repair_sha(protocol_path),"manifest_sha256":_repair_sha(manifest_path),
        "normalization":normalization,"class_order":REPAIR_NAMES,"class_cost_ratio":[1,1,2],
        "learning_rate":1e-5,"optimizer":"fresh Adam in both arms; original task-weight coefficients restored",
        "clipnorm":1.,"batch_size":8,"max_phase_epochs":12,"patience":4,
        "augmentation":"identical per-seed/epoch shuffling and horizontal flips (p=0.5) in both arms",
        "encoder_tuning":"last four leaf Conv2D/DepthwiseConv2D/SeparableConv2D layers only; all encoder BatchNorm frozen",
        "loss":"unchanged weighted CE + BCE/positive Dice with original learned task weights",
        "selection":["malignant_vs_other_balanced_accuracy","positive_Dice","macro_F1"],
        "initial_checkpoint_included":True,"test_evaluated":False,
        "comparison":"paired new frozen/unfrozen continuations from the same seed-specific selected original trainer",
        "limitation":"exploratory follow-up after earlier test inspection; no new independent test claim; GPU/dropout replay after interruption is not bit exact",
        "validation_improvement_guards":["mean balanced accuracy increases","mean malignant recall not lower","mean Dice not lower","mean normal false area not higher"],
        "versions":{"tensorflow":tf.__version__,"keras":tf.keras.__version__,"numpy":np.__version__,"python":platform.python_version()}}
    _repair_guard(out/"settings.json",settings)
    shutil.copyfile(out/"settings.json",report/"settings.json")
    if (out/"completed.json").is_file():
        saved=json.loads((out/"completed.json").read_text())
        for name in saved["report_files"]:
            if not (report/name).is_file():
                raise FileNotFoundError("Missing completed report: "+name)
        archive=shutil.make_archive("/content/BUSI_Targeted_Finetune_RESULTS","zip",root_dir=report)
        print("Completed experiment reused; no training rerun.")
        return {"directory":out,"comparison":pd.read_csv(report/"validation_comparison.csv"),
            "guards":pd.read_csv(report/"validation_improvement_guards.csv"),"zip_path":archive}
    data,frames=_read_data(source,manifest,Path(dataset_root),None)
    digest=hashlib.sha256()
    for key in ["X_train","M_train","y_train","X_val","M_val","y_val"]:
        array=np.ascontiguousarray(data[key])
        digest.update(str((key,array.shape,array.dtype.str)).encode())
        digest.update(memoryview(array).cast("B"))
    _repair_guard(out/"input_arrays.json",{"sha256":digest.hexdigest()})
    shutil.copyfile(out/"input_arrays.json",report/"input_arrays.json")
    cuts=np.quantile(data["M_train"].sum((1,2,3))[data["y_train"]!=0]/(224*224),[.25,.5,.75])
    rows,per_images,checks,reference_values=[],[],[],{}
    np.savez_compressed(report/"validation_reference.npz",images=frames["val"]["image"].to_numpy(dtype=str),labels=data["y_val"],
        mask_bits=np.packbits((data["M_val"][...,0]>0).reshape(115,-1),axis=1),mask_shape=np.array([224,224]))
    for item in items:
        print("Checking source model and source trainer, seed",item["seed"],flush=True)
        saved=tf.keras.models.load_model(item["trainer"],compile=False)
        best=tf.keras.models.load_model(item["model"],compile=False)
        a,b=saved.network.weights,best.weights
        if len(a)!=len(b):
            raise ValueError("Original trainer/network weight counts differ")
        for wa,wb in zip(a,b):
            np.testing.assert_array_equal(wa.numpy(),wb.numpy(),err_msg="Original best network and trainer differ")
        item["parameters"]=saved.network.count_params()
        values=_repair_prediction(saved.network,data)
        stats=validation_statistics(data["y_val"],data["M_val"],values)[0]
        for field in REPAIR_FIELDS:
            expected=item["validation"][field]
            checks.append({"seed":item["seed"],"metric":field,"observed":stats[field],"expected":expected,
                "matches":bool(np.isclose(stats[field],expected,atol=1e-5,rtol=0))})
        if not all(c["matches"] for c in checks):
            raise ValueError("Source validation predictions differ; no new training started")
        row,table=_repair_save_predictions(report/f"original_model_seed{item['seed']}",frames["val"],data,values,cuts,
            "original_model",item["seed"],item["selected_epoch"])
        row.update(source_epoch=item["selected_epoch"],selected_phase_epoch=0,initial_model_retained=True)
        _write_json(report/f"original_model_seed{item['seed']}"/"summary.json",row)
        rows.append(row);table.insert(0,"condition",row["condition"]);table.insert(1,"seed",item["seed"]);per_images.append(table)
        reference_values[item["seed"]]=values
        del saved,best,a,b
        gc.collect()
    pd.DataFrame(checks).to_csv(report/"original_validation_recheck.csv",index=False)
    if not (report/"colab_smoke_test.json").is_file():
        _phase_smoke(items[0],data,weights,report)
    for item in items:
        for mode in PHASE_MODES:
            print("Starting",mode,"seed",item["seed"],flush=True)
            row,table=_phase_run_one(out,report,item,mode,data,frames["val"],cuts,weights,reference_values[item["seed"]])
            rows.append(row);table.insert(0,"condition",row["condition"]);table.insert(1,"seed",item["seed"]);per_images.append(table)
            pd.DataFrame(rows).to_csv(report/"validation_comparison_progress.csv",index=False)
    summary,means,guards=_phase_final_tables(report,rows,per_images)
    _phase_figures(report,frames["val"],data,normalization)
    shutil.copytree(out/"paired_epoch_inputs",report/"paired_epoch_inputs",dirs_exist_ok=True)
    notes=("Paired low-LR continuation and limited encoder fine-tuning.\n"
        "Both arms start from the same seed-specific selected original trainer, including task coefficients.\n"
        "Both use fresh Adam(1e-5), identical image order/flips, identical loss and checkpoint selection.\n"
        "Only the last four encoder convolutions become trainable in the fine-tuning arm. Encoder BN and all other encoder weights stay frozen.\n"
        "Source epoch and phase epoch are different counters. Phase epoch 0 means the original model was retained.\n"
        "All comparisons are validation-only follow-ups after prior test inspection; do not describe them as an independent test.\n"
        "Passing the mean-metric guards is descriptive, not statistical proof of superiority. No guaranteed correction of individual cases.\n"
        "Models and optimizer checkpoints remain in Drive and are excluded from this report ZIP.\n"
        "Reference: https://keras.io/guides/transfer_learning/\n")
    (report/"README.txt").write_text(notes,encoding="utf-8")
    for item in items:
        if _repair_sha(item["model"])!=item["model_sha256"] or _repair_sha(item["trainer"])!=item["trainer_sha256"]:
            raise RuntimeError("An original checkpoint changed")
    _write_json(out/"completed.json",{"version":PHASE_VERSION,"test_evaluated":False,
        "report_files":[str(p.relative_to(report)) for p in sorted(report.rglob("*")) if p.is_file()]})
    archive=shutil.make_archive("/content/BUSI_Targeted_Finetune_RESULTS","zip",root_dir=report)
    print(means.to_string());print(guards.to_string(index=False));print("Saved:",out)
    return {"directory":out,"comparison":summary,"guards":guards,"zip_path":archive}


In [ ]:
#@title ۳. کنترل، ادامهٔ آموزش و دریافت گزارش
from IPython.display import display
from google.colab import files

phase_result = run_busi_targeted_finetune()

columns = ["condition", "seed", "source_epoch", "selected_phase_epoch",
           "val_accuracy", "val_malignant_recall", "val_malignant_balanced_accuracy",
           "val_positive_dice", "val_normal_mean_foreground_percent"]
display(phase_result["comparison"][columns])
display(phase_result["guards"])
print("Saved:", phase_result["directory"])
files.download(phase_result["zip_path"])


<div dir="rtl" align="right">
<p>پیش از آموزش اصلی، تطابق مدل‌های قبلی با گزارش validation، دریافت گرادیان در کانولوشن‌های بازشده، ثابت ماندن BatchNorm و ذخیره/بارگذاری optimizer در خود Colab کنترل می‌شوند.</p>
<p>در جدول، <code dir="ltr">source_epoch</code> شمارهٔ epoch مدل اصلی و <code dir="ltr">selected_phase_epoch</code> شمارهٔ epoch در این مرحله است. مقدار صفر در ستون دوم یعنی مدل اصلی حفظ شده است.</p>
<p>خروجی در پوشهٔ <code dir="ltr">targeted_finetune_v1</code> ذخیره می‌شود. اگر اجرا قطع شد، همین نوت‌بوک را دوباره اجرا کن تا از checkpoint آخر ادامه دهد. مدل‌های قبلی حفظ می‌شوند.</p>
<p>فایل دانلودی <code dir="ltr">BUSI_Targeted_Finetune_RESULTS.zip</code> را برای بررسی بفرست. این فایل شامل تمام نتایج validation، جدول قبل/بعد، کنترل‌های فنی و شکل‌های مقایسه‌ای است. checkpointهای مدل داخل Drive می‌مانند.</p>
<p>ساختار کد و محاسبات مستقل بررسی شده‌اند؛ اجرای واقعی TensorFlow روی وزن‌های Drive در Colab انجام می‌شود. این ادامهٔ اکتشافی پس از مشاهدهٔ test قبلی است و آزمون مستقل جدید محسوب نمی‌شود.</p>
<p>مرجع روش Fine-tuning: <a href="https://keras.io/guides/transfer_learning/">Keras transfer learning guide</a>.</p>
</div>
